In [2]:
# ============================================================================
# Notebook: 01_download.ipynb
#
# Verification Channels and Remote Work — Data acquisition & integrity check
#
# Purpose:
#   - Ensure the Stack Overflow Developer Survey 2024 & 2025 raw CSVs exist
#     under data/raw/ (download only if missing).
#   - Verify each file is a real CSV, not a tiny Git-LFS pointer.
#   - Run a light sanity check (shape + presence of the variables this study needs).
#
# Assumes this notebook runs from the notebooks/ folder, so the project root
# is one level up.
# ============================================================================


# %%
# ---------------------------------------------------------------------------
# Cell 1 | Paths & config
# ---------------------------------------------------------------------------
from pathlib import Path
import urllib.request
import pandas as pd

# Project root = parent of the notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Official SO survey data (Git-LFS backed GitHub repo).
# If a link ever yields an LFS pointer, use the "Data (CSV)" ZIP from
# https://survey.stackoverflow.co/ instead.
SOURCES = {
    2024: "https://github.com/StackExchange/Survey/raw/refs/heads/main/packages/archive/2024/results.csv",
    2025: "https://github.com/StackExchange/Survey/raw/refs/heads/main/packages/archive/2025/results.csv",
}

FILES = {year: RAW_DIR / f"survey_{year}.csv" for year in SOURCES}

print("Paths configured.")


# %%
# ---------------------------------------------------------------------------
# Cell 2 | Download only if missing
#
# A real results.csv is tens of MB. A few hundred bytes means the download
# returned a Git-LFS pointer rather than the actual data.
# ---------------------------------------------------------------------------
LFS_POINTER_MAX_BYTES = 1_000  # anything smaller is almost certainly a pointer

def looks_like_lfs_pointer(path: Path) -> bool:
    if path.stat().st_size > LFS_POINTER_MAX_BYTES:
        return False
    head = path.read_text(errors="ignore")[:200]
    return "git-lfs" in head

for year, url in SOURCES.items():
    dst = FILES[year]
    if dst.exists() and not looks_like_lfs_pointer(dst):
        size_mb = dst.stat().st_size / 1e6
        print(f"{year}: already present ({size_mb:.1f} MB) — skipping download")
        continue

    print(f"{year}: downloading ...")
    urllib.request.urlretrieve(url, dst)
    size_mb = dst.stat().st_size / 1e6
    if looks_like_lfs_pointer(dst):
        print(f"{year}: [ERROR] got an LFS pointer ({dst.stat().st_size} bytes). "
              f"Download the CSV ZIP manually from https://survey.stackoverflow.co/ "
              f"and place it in data/raw/ as survey_{year}.csv")
    else:
        print(f"{year}: downloaded ({size_mb:.1f} MB)")


# %%
# ---------------------------------------------------------------------------
# Cell 3 | Integrity check — file sizes
# ---------------------------------------------------------------------------
print("Raw files:")
for year, path in FILES.items():
    if path.exists():
        print(f"  survey_{year}.csv   {path.stat().st_size / 1e6:8.1f} MB")
    else:
        print(f"  survey_{year}.csv   MISSING")


# %%
# ---------------------------------------------------------------------------
# Cell 4 | Sanity check — load headers and confirm required variables exist
#
# We do NOT recode here (that is notebook 02). We only confirm the raw files
# load and contain the columns this study depends on.
# ---------------------------------------------------------------------------
REQUIRED_COLS = [
    "RemoteWork",   # moderator source
    "AISelect",     # AI use (and 2025 frequency)
    "JobSat",       # outcome (0-10)
    "Country",      # level-2 grouping
    "DevType",      # control (collapsed later)
    "WorkExp",      # control (continuous)
]

for year, path in FILES.items():
    if not path.exists():
        print(f"{year}: file missing — skip check")
        continue
    df = pd.read_csv(path, low_memory=False)
    present = [c for c in REQUIRED_COLS if c in df.columns]
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    print(f"\n{year}: shape = {df.shape}")
    print(f"   required present: {present}")
    if missing:
        print(f"   [WARN] required MISSING: {missing}")
    else:
        print(f"   all required columns present ✓")


# %%
# ---------------------------------------------------------------------------
# Cell 5 | Done
# ---------------------------------------------------------------------------
print("Data acquisition complete.")
print("Next: 02_recode.ipynb  (recode raw -> data/processed/ analytic samples)")

Paths configured.
2024: already present (159.5 MB) — skipping download
2025: already present (140.9 MB) — skipping download
Raw files:
  survey_2024.csv      159.5 MB
  survey_2025.csv      140.9 MB

2024: shape = (65437, 114)
   required present: ['RemoteWork', 'AISelect', 'JobSat', 'Country', 'DevType', 'WorkExp']
   all required columns present ✓

2025: shape = (49191, 172)
   required present: ['RemoteWork', 'AISelect', 'JobSat', 'Country', 'DevType', 'WorkExp']
   all required columns present ✓
Data acquisition complete.
Next: 02_recode.ipynb  (recode raw -> data/processed/ analytic samples)
